# Developmental modulation of narrative-detail compression

## Primary research question

**When the same full story is summarized for younger, actual-age, and older audiences, do (a) the amount of source information preserved and (b) the prioritization of narratively central details change—and does either condition effect vary with the source story's age level?**

The story is the inferential unit. The confirmatory cohort contains only stories with all three summary conditions.

## How to read the analyses

1. **Amount preserved — continuous coverage.** For every original detail, find its strongest semantic correspondence anywhere in the summary and average these scores within a story.
2. **Which details are prioritized — centrality–coverage correlation.** Within each story, correlate original-detail centrality with continuous coverage. Positive values mean central details are better represented than peripheral details.
3. **Representational reweighting.** Measure whether each original detail becomes more or less aligned with the whole summary than with the whole original.
4. **Developmental modulation.** Form younger-minus-actual and older-minus-actual contrasts within each story, then test whether those contrasts change with source age.
5. **Secondary layerwise readout.** EOS and mean-pooled whole-text representations show where across model depth original-to-summary geometry changes. They do not identify which details changed and are not the primary outcome.

Binary nearest-neighbor winner selection and one-to-one Hungarian matching are diagnostics, not the primary definition of retention.

In [ ]:
from pathlib import Path
import os

CACHE_ROOT = Path(os.environ.get('SLURM_TMPDIR', '/tmp')) / f'story-features-{os.getuid()}'
MPL_CACHE = CACHE_ROOT / 'matplotlib'
MPL_CACHE.mkdir(parents=True, exist_ok=True)
os.environ['MPLCONFIGDIR'] = str(MPL_CACHE)
os.environ.setdefault('MPLBACKEND', 'Agg')

ROOT = Path(os.environ.get('STORY_FEATURES_ROOT', Path.cwd())).resolve()
if not (ROOT / 'mechanistic_interpretability').exists():
    ROOT = ROOT.parent
MODEL = os.environ.get('REPRESENTATION_MODEL', 'gpt_oss_20b')
raw_defaults = {
    'gpt_oss_20b': ROOT / 'mechanistic_interpretability/outputs/02_representation_extraction/gpt_oss_20b_global_current',
    'qwen3_8b': ROOT / 'mechanistic_interpretability/outputs/02_representation_extraction/qwen3_8b_aug27_global',
}
RAW_DIR = Path(os.environ.get('REPRESENTATION_RAW_DIR', raw_defaults[MODEL]))
ANALYSIS_DIR = Path(os.environ.get(
    'REPRESENTATION_ANALYSIS_DIR',
    ROOT / 'mechanistic_interpretability/outputs/02_representation_extraction/analysis' / MODEL,
))

import numpy as np
import pandas as pd
from IPython.display import display

PLOTTING_AVAILABLE = True
STATSMODELS_AVAILABLE = True
try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    sns.set_theme(style='whitegrid', context='notebook')
except Exception as error:
    PLOTTING_AVAILABLE = False
    print('Plotting unavailable:', type(error).__name__, error)
try:
    import statsmodels.formula.api as smf
except Exception as error:
    STATSMODELS_AVAILABLE = False
    print('Statsmodels unavailable:', type(error).__name__, error)

def clustered_age_condition_regression(data, outcome, controls=()):
    from scipy.stats import t as student_t
    columns = [outcome, 'condition', 'source_age', 'story_key', *controls]
    frame = data.dropna(subset=columns).copy()
    frame['source_age_centered'] = frame['source_age'] - frame['source_age'].mean()
    design = pd.DataFrame({'Intercept': 1.0}, index=frame.index)
    for condition in ['older', 'younger']:
        indicator = frame['condition'].eq(condition).astype(float)
        design[f'condition_{condition}'] = indicator
        design[f'{condition}_x_source_age'] = indicator * frame['source_age_centered']
    design['source_age_centered'] = frame['source_age_centered']
    for control in controls:
        scale = frame[control].std()
        design[f'z_{control}'] = (frame[control] - frame[control].mean()) / (scale if scale else 1.0)
    x = design.to_numpy(float)
    y = frame[outcome].to_numpy(float)
    beta = np.linalg.pinv(x.T @ x) @ x.T @ y
    residual = y - x @ beta
    bread = np.linalg.pinv(x.T @ x)
    meat = np.zeros((x.shape[1], x.shape[1]))
    groups = frame['story_key'].astype(str).to_numpy()
    unique_groups = np.unique(groups)
    for group in unique_groups:
        use = groups == group
        score = x[use].T @ residual[use]
        meat += np.outer(score, score)
    n, k, g = len(frame), x.shape[1], len(unique_groups)
    correction = (g / (g - 1)) * ((n - 1) / (n - k)) if g > 1 and n > k else 1.0
    covariance = correction * bread @ meat @ bread
    standard_error = np.sqrt(np.maximum(np.diag(covariance), 0))
    statistic = beta / np.maximum(standard_error, 1e-12)
    p_value = 2 * student_t.sf(np.abs(statistic), df=max(g - 1, 1))
    return pd.DataFrame({
        'term': design.columns, 'estimate': beta, 'clustered_se': standard_error,
        't': statistic, 'p_value': p_value, 'clusters': g, 'observations': n,
    })

print('repository:', ROOT)
print('model:', MODEL)
print('raw representations:', RAW_DIR)
print('analysis:', ANALYSIS_DIR)

## Representation-level omnibus test in an original-story PCA basis

PCA is used only as an unsupervised coordinate system, not as a search over significant components. At each layer, the PCA basis is fitted to original-story EOS vectors without condition or age labels; all summaries are then projected into that fixed basis. The prespecified representation statistic is the sum of squared studentized effects across the first `PCA_COMPONENTS` coordinates. No individual PC is selected or interpreted.

Two block-permutation tests are reported. The condition test permutes the three condition labels within each story using the same permutation across layers, then takes the maximum statistic over layers and contrasts. The developmental test keeps each triplet intact, permutes source ages across stories, and takes the maximum over layers. Layer curves localize a representation-level effect only if the corresponding omnibus test rejects.

In [ ]:
from sklearn.decomposition import PCA

PCA_COMPONENTS = int(os.environ.get('REPRESENTATION_PCA_COMPONENTS', 32))
PCA_PERMUTATIONS = int(os.environ.get('REPRESENTATION_PCA_PERMUTATIONS', 2000))
feature_parts, metadata_parts, raw_layers = [], [], None
offset = 0
for feature_path in sorted(RAW_DIR.glob('global_features_*.npz')):
    metadata_path = RAW_DIR / feature_path.name.replace('global_features_', 'global_metadata_').replace('.npz', '.csv')
    if not metadata_path.exists():
        continue
    payload = np.load(feature_path, allow_pickle=False)
    shard_layers = payload['layers'].astype(int)
    if raw_layers is None:
        raw_layers = shard_layers
    elif not np.array_equal(raw_layers, shard_layers):
        raise ValueError(f'Layer mismatch in {feature_path}')
    vectors = payload['eos_vectors'].astype(np.float32)
    metadata = pd.read_csv(metadata_path, low_memory=False)
    if len(vectors) != len(metadata):
        raise ValueError(f'Feature/metadata mismatch in {feature_path}')
    metadata['pca_row'] = np.arange(len(metadata)) + offset
    offset += len(metadata)
    feature_parts.append(vectors)
    metadata_parts.append(metadata)
if not feature_parts:
    raise FileNotFoundError(f'No complete global feature shards in {RAW_DIR}')
pca_vectors = np.concatenate(feature_parts)
pca_metadata = pd.concat(metadata_parts, ignore_index=True)
pca_metadata['source_age'] = pd.to_numeric(pca_metadata['source_age'], errors='coerce')
pca_metadata['target_age'] = pd.to_numeric(pca_metadata['target_age'], errors='coerce')
original_metadata = pca_metadata[pca_metadata['record_type'].eq('original')].drop_duplicates('story_key')
all_summary_metadata = pca_metadata[pca_metadata['record_type'].eq('summary')].copy()
pca_condition_sets = {condition: set(all_summary_metadata.loc[all_summary_metadata['condition'].eq(condition), 'story_key'])
                      for condition in ['younger', 'actual', 'older']}
pca_complete_story_ids = set.intersection(*pca_condition_sets.values())
summary_metadata = (all_summary_metadata[all_summary_metadata['story_key'].isin(pca_complete_story_ids)]
                    .sort_values('summary_id').drop_duplicates(['story_key', 'condition']))
pca_component_count = min(PCA_COMPONENTS, len(original_metadata) - 1, pca_vectors.shape[-1])
pca_coordinates = {}
pca_variance_rows = []
pca_distance_rows = []
original_rows = original_metadata['pca_row'].astype(int).to_numpy()
original_row_by_story = original_metadata.set_index('story_key')['pca_row'].astype(int).to_dict()
for layer_position, layer in enumerate(raw_layers):
    estimator = PCA(n_components=pca_component_count, svd_solver='randomized', random_state=20260902)
    estimator.fit(pca_vectors[original_rows, layer_position])
    selected_rows = np.unique(np.concatenate([original_rows, summary_metadata['pca_row'].astype(int).to_numpy()]))
    coordinates = estimator.transform(pca_vectors[selected_rows, layer_position])
    coordinate_by_row = {int(row): coordinates[index] for index, row in enumerate(selected_rows)}
    pca_coordinates[int(layer)] = coordinate_by_row
    pca_variance_rows.append({'layer': int(layer), 'components': pca_component_count,
                              'explained_variance_fraction': estimator.explained_variance_ratio_.sum()})
    for _, summary_row in summary_metadata.iterrows():
        original_row = original_row_by_story.get(summary_row['story_key'])
        if original_row is None:
            continue
        displacement = coordinate_by_row[int(summary_row['pca_row'])] - coordinate_by_row[int(original_row)]
        pca_distance_rows.append({
            'story_key': summary_row['story_key'], 'condition': summary_row['condition'],
            'source_age': summary_row['source_age'], 'target_age': summary_row['target_age'],
            'layer': int(layer), 'pca_displacement_norm': np.linalg.norm(displacement),
        })
pca_variance = pd.DataFrame(pca_variance_rows)
pca_distances = pd.DataFrame(pca_distance_rows)
display(pca_variance)
display(pca_distances.groupby(['layer', 'condition'], as_index=False).agg(
    stories=('story_key', 'nunique'), mean_displacement=('pca_displacement_norm', 'mean')))

In [ ]:
def multivariate_studentized_energy(vectors):
    vectors = np.asarray(vectors, float)
    standard_error = np.std(vectors, axis=0, ddof=1) / np.sqrt(len(vectors))
    usable = np.isfinite(standard_error) & (standard_error > 0)
    if not np.any(usable):
        return np.nan
    statistics = np.mean(vectors[:, usable], axis=0) / standard_error[usable]
    return float(np.sum(statistics ** 2))

condition_order = ['younger', 'actual', 'older']
pca_story_keys = sorted(pca_complete_story_ids)
summary_lookup = summary_metadata.set_index(['story_key', 'condition'])['pca_row'].astype(int).to_dict()
target_lookup = summary_metadata.set_index(['story_key', 'condition'])['target_age'].to_dict()
source_lookup = summary_metadata.groupby('story_key')['source_age'].mean().to_dict()
layer_condition_coordinates = {}
valid_story_keys = []
for story_key in pca_story_keys:
    if all((story_key, condition) in summary_lookup for condition in condition_order):
        valid_story_keys.append(story_key)
for layer in raw_layers:
    layer_condition_coordinates[int(layer)] = np.stack([
        np.stack([pca_coordinates[int(layer)][summary_lookup[(story_key, condition)]]
                  for condition in condition_order])
        for story_key in valid_story_keys])
observed_rows = []
contrast_functions = {
    'older - younger': lambda array: array[:, 2] - array[:, 0],
    'actual - flank mean': lambda array: array[:, 1] - array[:, [0, 2]].mean(axis=1),
}
for layer in raw_layers:
    for contrast, function in contrast_functions.items():
        observed_rows.append({'layer': int(layer), 'contrast': contrast,
                              'multivariate_energy': multivariate_studentized_energy(
                                  function(layer_condition_coordinates[int(layer)]))})
pca_condition_tests = pd.DataFrame(observed_rows)
rng = np.random.default_rng(20260902)
condition_null_max = np.empty(PCA_PERMUTATIONS)
for permutation_index in range(PCA_PERMUTATIONS):
    orders = np.argsort(rng.random((len(valid_story_keys), 3)), axis=1)
    statistics = []
    for layer in raw_layers:
        coordinates = layer_condition_coordinates[int(layer)]
        shuffled = np.take_along_axis(coordinates, np.broadcast_to(orders[:, :, None], coordinates.shape), axis=1)
        statistics.extend(multivariate_studentized_energy(function(shuffled))
                          for function in contrast_functions.values())
    condition_null_max[permutation_index] = np.nanmax(statistics)
pca_condition_omnibus = (1 + np.sum(condition_null_max >= pca_condition_tests['multivariate_energy'].max())) / (PCA_PERMUTATIONS + 1)
pca_condition_tests['p_fwer_maxT'] = [
    (1 + np.sum(condition_null_max >= statistic)) / (PCA_PERMUTATIONS + 1)
    for statistic in pca_condition_tests['multivariate_energy']]
print('PCA multivariate condition omnibus p:', pca_condition_omnibus)
display(pca_condition_tests)

source_ages = np.array([source_lookup[key] for key in valid_story_keys], float)
target_spans = np.array([target_lookup[(key, 'older')] - target_lookup[(key, 'younger')]
                         for key in valid_story_keys], float)
def multivariate_age_energy(age, response_vectors):
    age = np.asarray(age, float)
    centered_age = age - np.mean(age)
    design_scale = np.sum(centered_age ** 2)
    slopes = centered_age @ response_vectors / design_scale
    residuals = response_vectors - np.mean(response_vectors, axis=0) - np.outer(centered_age, slopes)
    slope_se = np.sqrt(np.sum(residuals ** 2, axis=0) / max(len(age) - 2, 1) / design_scale)
    usable = np.isfinite(slope_se) & (slope_se > 0)
    return float(np.sum((slopes[usable] / slope_se[usable]) ** 2))
local_vectors = {int(layer): (coordinates[:, 2] - coordinates[:, 0]) / target_spans[:, None]
                 for layer, coordinates in layer_condition_coordinates.items()}
pca_age_tests = pd.DataFrame([
    {'layer': int(layer), 'multivariate_source_age_energy': multivariate_age_energy(source_ages, local_vectors[int(layer)])}
    for layer in raw_layers])
age_null_max = np.empty(PCA_PERMUTATIONS)
for permutation_index in range(PCA_PERMUTATIONS):
    permuted_ages = rng.permutation(source_ages)
    age_null_max[permutation_index] = max(
        multivariate_age_energy(permuted_ages, local_vectors[int(layer)]) for layer in raw_layers)
pca_age_omnibus = (1 + np.sum(age_null_max >= pca_age_tests['multivariate_source_age_energy'].max())) / (PCA_PERMUTATIONS + 1)
pca_age_tests['p_fwer_maxT'] = [
    (1 + np.sum(age_null_max >= statistic)) / (PCA_PERMUTATIONS + 1)
    for statistic in pca_age_tests['multivariate_source_age_energy']]
print('PCA multivariate source-age modulation omnibus p:', pca_age_omnibus)
display(pca_age_tests)
if PLOTTING_AVAILABLE:
    fig, axes = plt.subplots(1, 2, figsize=(15, 5), constrained_layout=True)
    sns.lineplot(data=pca_condition_tests, x='layer', y='multivariate_energy', hue='contrast', marker='o', ax=axes[0])
    sns.lineplot(data=pca_age_tests, x='layer', y='multivariate_source_age_energy', marker='o', ax=axes[1])
    axes[0].set_title(f'Condition effect localization (omnibus p={pca_condition_omnibus:.3g})')
    axes[1].set_title(f'Source-age modulation localization (omnibus p={pca_age_omnibus:.3g})')
    plt.show()

## Immediate preview from raw metadata

This section works before the representation reducer finishes. It audits usable stories, steering-condition balance, age coverage, token compression, and detail compression. These are behavioral/QC results, not yet tests of hidden-state geometry.

In [ ]:
global_files = sorted(RAW_DIR.glob('global_metadata_*.csv'))
detail_files = sorted(RAW_DIR.glob('contextual_detail_metadata_*.csv'))
if not global_files or not detail_files:
    raise FileNotFoundError(f'Raw metadata not found in {RAW_DIR}')
global_meta = pd.concat([pd.read_csv(path, low_memory=False) for path in global_files], ignore_index=True)
detail_meta = pd.concat([pd.read_csv(path, low_memory=False) for path in detail_files], ignore_index=True)
for frame in [global_meta, detail_meta]:
    frame['source_age'] = pd.to_numeric(frame.get('source_age'), errors='coerce')
    frame['target_age'] = pd.to_numeric(frame.get('target_age'), errors='coerce')
original_meta = global_meta[global_meta['record_type'].eq('original')].drop_duplicates('story_key')
summary_meta = global_meta[global_meta['record_type'].eq('summary')].copy()
record_detail_counts = detail_meta.groupby('record_id').size().rename('n_details')
paired_meta = summary_meta.merge(
    original_meta[['story_key', 'record_id', 'token_count']].rename(
        columns={'record_id': 'original_record_id', 'token_count': 'original_token_count'}),
    on='story_key', how='inner', validate='many_to_one')
paired_meta = paired_meta.rename(columns={'token_count': 'summary_token_count'})
paired_meta = paired_meta.join(record_detail_counts, on='record_id').rename(columns={'n_details': 'n_summary_details'})
paired_meta = paired_meta.join(record_detail_counts, on='original_record_id').rename(columns={'n_details': 'n_original_details'})
paired_meta['token_ratio'] = paired_meta['summary_token_count'] / paired_meta['original_token_count']
paired_meta['detail_ratio'] = paired_meta['n_summary_details'] / paired_meta['n_original_details']
paired_meta['age_shift'] = paired_meta['target_age'] - paired_meta['source_age']
print('global records:', len(global_meta), 'stories:', original_meta['story_key'].nunique(), 'summaries:', len(summary_meta))
print('contextual details:', len(detail_meta))
display(paired_meta.groupby('condition').agg(
    stories=('story_key', 'nunique'), summaries=('record_id', 'nunique'),
    median_token_ratio=('token_ratio', 'median'), median_detail_ratio=('detail_ratio', 'median')).reset_index())

In [ ]:
if PLOTTING_AVAILABLE:
    fig, axes = plt.subplots(2, 2, figsize=(16, 11))
    sns.countplot(data=paired_meta, x='condition', order=['younger', 'actual', 'older'], ax=axes[0, 0])
    axes[0, 0].set_title('Available summaries by condition')
    sns.histplot(data=paired_meta, x='source_age', hue='condition', multiple='layer', discrete=True, ax=axes[0, 1])
    axes[0, 1].set_title('Source-age support')
    sns.boxplot(data=paired_meta, x='condition', y='token_ratio', order=['younger', 'actual', 'older'], ax=axes[1, 0])
    axes[1, 0].set_title('Token compression ratio')
    sns.boxplot(data=paired_meta, x='condition', y='detail_ratio', order=['younger', 'actual', 'older'], ax=axes[1, 1])
    axes[1, 1].set_title('Detail-count compression ratio')
    plt.tight_layout()
else:
    display(paired_meta.groupby('condition')[['token_ratio', 'detail_ratio']].describe())

support = (paired_meta.assign(available=1)
           .pivot_table(index='story_key', columns='condition', values='available', aggfunc='max', fill_value=0))
print('paired actual + older stories:', int(((support.get('actual', 0) == 1) & (support.get('older', 0) == 1)).sum()))
print('paired actual + younger stories:', int(((support.get('actual', 0) == 1) & (support.get('younger', 0) == 1)).sum()))

### Statistical QC available immediately

The cluster-robust regressions below test whether output length and number of details differ by steering condition across source age. Because summary length was prompted to approximately 30%, these are manipulation checks rather than the paper's central representational tests.

In [ ]:
metadata_models = {}
if STATSMODELS_AVAILABLE:
    for outcome in ['token_ratio', 'detail_ratio']:
        frame = paired_meta.dropna(subset=[outcome, 'source_age', 'condition']).copy()
        fit = smf.ols(f'{outcome} ~ C(condition) * source_age', data=frame).fit(
            cov_type='cluster', cov_kwds={'groups': frame['story_key']})
        metadata_models[outcome] = fit
        print('\n', outcome)
        display(fit.summary2().tables[1])
else:
    print('Using the NumPy/SciPy story-clustered OLS fallback.')
    for outcome in ['token_ratio', 'detail_ratio']:
        metadata_models[outcome] = clustered_age_condition_regression(paired_meta, outcome)
        print('\n', outcome)
        display(metadata_models[outcome])

## Load compact HPC results

The raw hidden states remain on HPC. The notebook only needs four compact tables produced by the analysis commands in the final section.

In [ ]:
paths = {
    'whole': ANALYSIS_DIR / 'story_summary_global_geometry.csv',
    'story': ANALYSIS_DIR / 'story_representation_compression_metrics.csv',
    'source_details': ANALYSIS_DIR / 'original_detail_representation_coverage.csv.gz',
    'summary_details': ANALYSIS_DIR / 'summary_detail_representation_alignment.csv.gz',
}
missing = [str(path) for path in paths.values() if not path.exists()]
if missing:
    raise FileNotFoundError('Run the HPC reduction commands first. Missing:\n' + '\n'.join(missing))
whole = pd.read_csv(paths['whole'], low_memory=False)
story = pd.read_csv(paths['story'], low_memory=False)
source_details = pd.read_csv(paths['source_details'], low_memory=False)
summary_details = pd.read_csv(paths['summary_details'], low_memory=False)
ratio_columns = paired_meta[['record_id', 'token_ratio', 'detail_ratio']].drop_duplicates('record_id')
story = story.merge(ratio_columns, on='record_id', how='left', validate='one_to_one')
for frame in [whole, story, source_details, summary_details]:
    for column in ['source_age', 'target_age']:
        if column in frame:
            frame[column] = pd.to_numeric(frame[column], errors='coerce')
    if {'source_age', 'target_age'} <= set(frame):
        frame['age_shift'] = frame['target_age'] - frame['source_age']
print({name: len(frame) for name, frame in {
    'whole': whole, 'story': story, 'source_details': source_details, 'summary_details': summary_details
}.items()})

## Cohort audit

Before hypothesis testing, verify the number of stories, summaries, conditions, ages, and represented layers. Younger summaries are expected to be less frequent, so uncertainty—not raw means alone—must guide interpretation.

In [ ]:
required_conditions = {'younger', 'actual', 'older'}
condition_sets = {condition: set(story.loc[story['condition'].eq(condition), 'story_key'])
                  for condition in required_conditions}
complete_story_ids = set.intersection(*condition_sets.values())
story_cc = story[story['story_key'].isin(complete_story_ids)].copy()
whole_cc = whole[whole['story_key'].isin(complete_story_ids)].copy()
source_details_cc = source_details[source_details['story_key'].isin(complete_story_ids)].copy()
summary_details_cc = summary_details[summary_details['story_key'].isin(complete_story_ids)].copy()
print('complete-case stories with younger + actual + older:', len(complete_story_ids))
print('complete-case summary records:', story_cc['record_id'].nunique())
audit = (story_cc.groupby('condition', dropna=False)
         .agg(stories=('story_key', 'nunique'), summaries=('record_id', 'nunique'),
              source_age_min=('source_age', 'min'), source_age_max=('source_age', 'max'),
              target_age_min=('target_age', 'min'), target_age_max=('target_age', 'max'))
         .reset_index())
display(audit)
display(pd.crosstab(story_cc['source_age'], story_cc['condition']))
if not complete_story_ids:
    raise ValueError('No stories have all three age-steering conditions in the reduced outputs.')

## Secondary analysis — where across layers does whole-text displacement change?

**EOS representation** is the hidden state of the final token after it has causally attended to the preceding full text. **Mean pooling** averages all token hidden states at a layer and serves as a distributed-content control. Neither is an embedding of the segmented details.

Different values across layers mean that original and summary texts become differently organized as successive transformer blocks contextualize them. A layer decline by itself is expected for different texts and is not an age effect. The relevant age quantities are paired `older − actual` and `younger − actual` contrasts, and whether those contrasts vary with source age.

In [ ]:
from scipy.stats import wilcoxon

def bh_adjust(values):
    values = np.asarray(values, dtype=float)
    adjusted = np.full(len(values), np.nan)
    valid = np.flatnonzero(np.isfinite(values))
    if not len(valid):
        return adjusted
    order = valid[np.argsort(values[valid])]
    ranked = values[order] * len(order) / np.arange(1, len(order) + 1)
    ranked = np.minimum.accumulate(ranked[::-1])[::-1]
    adjusted[order] = np.clip(ranked, 0, 1)
    return adjusted

whole_profile = (whole_cc.groupby(['layer', 'condition'], as_index=False)
                 .agg(stories=('story_key', 'nunique'), eos_cosine=('eos_cosine', 'mean'),
                      mean_control_cosine=('mean_control_cosine', 'mean')))
display(whole_profile)
layer_difference_rows = []
for metric in ['eos_cosine', 'mean_control_cosine']:
    wide = whole_cc.pivot_table(index=['story_key', 'layer'], columns='condition', values=metric, aggfunc='mean')
    for comparison in ['younger', 'older']:
        pair = wide[['actual', comparison]].dropna()
        difference = pair[comparison] - pair['actual']
        for (story_key, layer), value in difference.items():
            layer_difference_rows.append({'story_key': story_key, 'layer': layer, 'metric': metric,
                                          'contrast': f'{comparison} - actual', 'difference': value})
layer_differences = pd.DataFrame(layer_difference_rows)
layer_tests = []
for keys, frame in layer_differences.groupby(['metric', 'layer', 'contrast']):
    values = frame['difference'].dropna().to_numpy()
    statistic, p_value = wilcoxon(values) if len(values) and np.any(values != 0) else (np.nan, np.nan)
    layer_tests.append({'metric': keys[0], 'layer': keys[1], 'contrast': keys[2],
                        'n_stories': len(values), 'mean_difference': np.mean(values),
                        'median_difference': np.median(values), 'p_value': p_value})
layer_tests = pd.DataFrame(layer_tests)
layer_tests = layer_tests.rename(columns={'p_value': 'exploratory_p_value_not_for_selection'})
display(layer_tests.sort_values(['metric', 'layer', 'contrast']))
if PLOTTING_AVAILABLE:
    fig, axes = plt.subplots(2, 2, figsize=(16, 11))
    sns.lineplot(data=whole_profile, x='layer', y='eos_cosine', hue='condition', marker='o', ax=axes[0, 0])
    sns.lineplot(data=whole_profile, x='layer', y='mean_control_cosine', hue='condition', marker='o', ax=axes[0, 1])
    for ax, metric in zip(axes[1], ['eos_cosine', 'mean_control_cosine']):
        sns.lineplot(data=layer_differences[layer_differences['metric'].eq(metric)],
                     x='layer', y='difference', hue='contrast', marker='o', errorbar=('ci', 95), ax=ax)
        ax.axhline(0, color='black', linestyle='--', linewidth=1)
        ax.set_title(f'Paired condition effect: {metric}')
    axes[0, 0].set_title('Original-to-summary EOS similarity')
    axes[0, 1].set_title('Original-to-summary mean-pooling similarity')
    plt.tight_layout()

### Randomization control for the three age conditions

Different summaries must have different representations. The relevant null is therefore not zero original-to-summary distance. Instead, within every story, randomly permute the `younger`, `actual`, and `older` labels across the three observed summaries using the **same permutation at every layer**. This preserves each story's three texts and complete layer trajectories while destroying only their association with target age. Two prespecified statistics are tested: `older - younger` and `actual - mean(younger, older)`. The primary decision is one max-|T| omnibus test across all layers and both contrasts; layer-specific familywise-adjusted values only localize an effect after that omnibus test rejects.

In [ ]:
def paired_t(values):
    values = np.asarray(values, float)
    values = values[np.isfinite(values)]
    if len(values) < 3 or np.std(values, ddof=1) == 0:
        return np.nan
    return np.mean(values) / (np.std(values, ddof=1) / np.sqrt(len(values)))

def layer_condition_permutation(frame, metric, permutations=5000, seed=20260902):
    rng = np.random.default_rng(seed)
    condition_order = ['younger', 'actual', 'older']
    layers = np.sort(frame['layer'].dropna().unique())
    wide = frame.pivot_table(index='story_key', columns=['layer', 'condition'], values=metric, aggfunc='mean')
    columns = pd.MultiIndex.from_product([layers, condition_order], names=['layer', 'condition'])
    wide = wide.reindex(columns=columns).dropna()
    values = wide.to_numpy(float).reshape(len(wide), len(layers), len(condition_order))
    if not len(values):
        return pd.DataFrame()
    def contrasts(array):
        return np.stack([array[:, :, 2] - array[:, :, 0],
                         array[:, :, 1] - array[:, :, [0, 2]].mean(axis=-1)], axis=-1)
    observed_differences = contrasts(values)
    observed_t = np.apply_along_axis(paired_t, 0, observed_differences)
    null_effects = np.empty((permutations, len(layers), 2))
    null_t = np.empty_like(null_effects)
    for permutation_index in range(permutations):
        orders = np.argsort(rng.random((len(values), 3)), axis=1)
        orders = np.broadcast_to(orders[:, None, :], values.shape)
        differences = contrasts(np.take_along_axis(values, orders, axis=2))
        null_effects[permutation_index] = np.mean(differences, axis=0)
        null_t[permutation_index] = np.apply_along_axis(paired_t, 0, differences)
    max_abs_null = np.nanmax(np.abs(null_t), axis=(1, 2))
    omnibus_observed = np.nanmax(np.abs(observed_t))
    omnibus_p = (1 + np.sum(max_abs_null >= omnibus_observed)) / (permutations + 1)
    names = ['older - younger', 'actual - flank mean']
    rows = []
    for layer_index, layer in enumerate(layers):
        for contrast_index, name in enumerate(names):
            statistic = observed_t[layer_index, contrast_index]
            effect = np.mean(observed_differences[:, layer_index, contrast_index])
            local_null = null_t[:, layer_index, contrast_index]
            rows.append({
                'metric': metric, 'layer': layer, 'statistic': name, 'n_stories': len(values),
                'observed': effect, 'observed_t': statistic,
                'null_low': np.quantile(null_effects[:, layer_index, contrast_index], .025),
                'null_high': np.quantile(null_effects[:, layer_index, contrast_index], .975),
                'permutation_p_uncorrected': (1 + np.sum(np.abs(local_null) >= abs(statistic))) / (permutations + 1),
                'p_fwer_maxT': (1 + np.sum(max_abs_null >= abs(statistic))) / (permutations + 1),
                'omnibus_p_maxT': omnibus_p,
            })
    return pd.DataFrame(rows)

layer_randomization = pd.concat([
    layer_condition_permutation(whole_cc, metric)
    for metric in ['eos_cosine', 'mean_control_cosine']
], ignore_index=True)
omnibus_tests = layer_randomization.groupby('metric', as_index=False).agg(
    omnibus_p_maxT=('omnibus_p_maxT', 'first'))
omnibus_tests['p_across_metrics_bh'] = bh_adjust(omnibus_tests['omnibus_p_maxT'])
display(omnibus_tests)
display(layer_randomization.sort_values(['metric', 'statistic', 'layer']))
if PLOTTING_AVAILABLE:
    fig, axes = plt.subplots(2, 2, figsize=(16, 11), sharex=True)
    for ax, ((metric, statistic), frame) in zip(axes.flat, layer_randomization.groupby(['metric', 'statistic'])):
        frame = frame.sort_values('layer')
        ax.fill_between(frame['layer'], frame['null_low'], frame['null_high'], alpha=.2, label='permuted 95% null')
        ax.plot(frame['layer'], frame['observed'], marker='o', label='observed')
        ax.axhline(0, color='black', linestyle='--', linewidth=1)
        ax.set_title(f'{metric}: {statistic}')
        ax.legend()
    plt.tight_layout()

### Does layerwise displacement vary with source age?

This is the developmentally relevant whole-text test. For every story and layer, calculate the paired condition contrast, then relate that contrast to source age. Source-age labels are permuted across intact story triplets using one shared permutation across every layer, metric, and contrast. The primary result is the max-|T| omnibus p-value; layer-specific adjusted values are localization only. Heatmaps show effect sizes rather than defining additional tests. The available complete-case source ages span only 5.5–10 years, so trends outside that interval are not identified.

In [ ]:
from scipy.stats import linregress, rankdata

story_age_lookup = (story_cc[['story_key', 'source_age']].dropna()
                    .drop_duplicates('story_key').set_index('story_key')['source_age'])
layer_age = layer_differences.copy()
layer_age['source_age'] = layer_age['story_key'].map(story_age_lookup)
age_story_keys = np.array(sorted(story_age_lookup.index.astype(str)))
age_values = story_age_lookup.reindex(age_story_keys).to_numpy(float)
age_positions = {story_key: index for index, story_key in enumerate(age_story_keys)}
def spearman_t(age, values):
    usable = np.isfinite(age) & np.isfinite(values)
    age, values = np.asarray(age)[usable], np.asarray(values)[usable]
    if len(age) < 4 or np.std(age) == 0 or np.std(values) == 0:
        return np.nan, np.nan
    rho = np.corrcoef(rankdata(age), rankdata(values))[0, 1]
    statistic = rho * np.sqrt((len(age) - 2) / max(1 - rho ** 2, 1e-12))
    return rho, statistic
layer_age_tests = []
layer_age_groups = []
for keys, frame in layer_age.dropna(subset=['source_age', 'difference']).groupby(['metric', 'layer', 'contrast']):
    fit = linregress(frame['source_age'], frame['difference'])
    row_positions = np.array([age_positions[str(key)] for key in frame['story_key']])
    differences = frame['difference'].to_numpy(float)
    rho, observed_t = spearman_t(age_values[row_positions], differences)
    layer_age_tests.append({
        'metric': keys[0], 'layer': keys[1], 'contrast': keys[2], 'n_stories': len(frame),
        'change_per_source_year': fit.slope, 'slope_se': fit.stderr,
        'spearman_rho': rho, 'observed_t': observed_t,
    })
    layer_age_groups.append((row_positions, differences))
layer_age_tests = pd.DataFrame(layer_age_tests)
rng = np.random.default_rng(20260902)
permutations = 5000
null_t = np.full((permutations, len(layer_age_groups)), np.nan)
for permutation_index in range(permutations):
    permuted_ages = rng.permutation(age_values)
    for test_index, (row_positions, differences) in enumerate(layer_age_groups):
        _, null_t[permutation_index, test_index] = spearman_t(permuted_ages[row_positions], differences)
max_abs_null = np.nanmax(np.abs(null_t), axis=1)
omnibus_observed = np.nanmax(np.abs(layer_age_tests['observed_t']))
layer_source_age_omnibus_p = (1 + np.sum(max_abs_null >= omnibus_observed)) / (permutations + 1)
layer_age_tests['permutation_p_uncorrected'] = [
    (1 + np.sum(np.abs(null_t[:, index]) >= abs(value))) / (permutations + 1)
    for index, value in enumerate(layer_age_tests['observed_t'])]
layer_age_tests['p_fwer_maxT'] = [
    (1 + np.sum(max_abs_null >= abs(value))) / (permutations + 1)
    for value in layer_age_tests['observed_t']]
print('Omnibus source-age modulation across every layer/metric/contrast:', layer_source_age_omnibus_p)
display(layer_age_tests.sort_values(['metric', 'contrast', 'layer']))
if PLOTTING_AVAILABLE:
    fig, axes = plt.subplots(2, 2, figsize=(17, 11), constrained_layout=True)
    combinations = [
        ('eos_cosine', 'older - actual'), ('eos_cosine', 'younger - actual'),
        ('mean_control_cosine', 'older - actual'), ('mean_control_cosine', 'younger - actual'),
    ]
    for ax, (metric, contrast) in zip(axes.flat, combinations):
        frame = layer_age[layer_age['metric'].eq(metric) & layer_age['contrast'].eq(contrast)]
        heatmap = frame.pivot_table(index='layer', columns='source_age', values='difference', aggfunc='mean')
        bound = np.nanmax(np.abs(heatmap.to_numpy()))
        sns.heatmap(heatmap, center=0, vmin=-bound, vmax=bound, cmap='coolwarm', ax=ax,
                    cbar_kws={'label': 'paired cosine difference'})
        ax.set_title(f'{metric}: {contrast}')
        ax.set_xlabel('Source-age midpoint')
        ax.set_ylabel('Model layer')
    plt.show()
    slope_plot = layer_age_tests.copy()
    slope_plot['low'] = slope_plot['change_per_source_year'] - 1.96 * slope_plot['slope_se']
    slope_plot['high'] = slope_plot['change_per_source_year'] + 1.96 * slope_plot['slope_se']
    fig, axes = plt.subplots(1, 2, figsize=(15, 5), constrained_layout=True)
    for ax, metric in zip(axes, ['eos_cosine', 'mean_control_cosine']):
        for contrast, frame in slope_plot[slope_plot['metric'].eq(metric)].groupby('contrast'):
            frame = frame.sort_values('layer')
            ax.plot(frame['layer'], frame['change_per_source_year'], marker='o', label=contrast)
            ax.fill_between(frame['layer'], frame['low'], frame['high'], alpha=.15)
        ax.axhline(0, color='black', linestyle='--', linewidth=1)
        ax.set_title(f'Source-age modulation by layer: {metric}')
        ax.set_xlabel('Model layer')
        ax.set_ylabel('Contrast change per source-age year')
        ax.legend()
    plt.show()

## Primary Q1 — how much original-detail content is preserved, and which details?

Let $O \in \mathbb{R}^{n \times d}$ contain normalized original-detail representations and $S \in \mathbb{R}^{m \times d}$ contain normalized summary-detail representations. The cross-text similarity matrix is $C = SO^\top$.

- **Summary grounding:** each summary detail $i$ selects $\arg\max_j C_{ij}$. Every summary detail is grounded, multiple summary details may select the same source, and there is no threshold.
- **Continuous coverage:** each original detail $j$ receives $r_j=\max_i C_{ij}$. It measures how strongly that source detail is represented anywhere in the summary without imposing a retained/not-retained threshold. `mean_coverage_cosine` averages $r_j$ over all source details in a story.
- **Original centrality:** $c_j$ is the mean cosine similarity of source detail $j$ to the other source details in the same story. High values identify details near the story's semantic center; low values identify comparatively distinctive or peripheral details.
- **Centrality–coverage correlation:** correlate $c_j$ with $r_j$ across source details within a story-summary pair. Positive values mean central details receive stronger coverage; zero means preservation is unrelated to centrality; negative values mean peripheral details receive stronger coverage.
- **NN-selected / not NN-selected:** an original detail is operationally selected when at least one summary detail chooses it as its nearest source. This is the threshold-free binary analysis called `selected_by_any_summary_detail`; it should not be interpreted as proof of semantic entailment.
- **Unique robustness check:** Hungarian matching maximizes total cosine subject to one-to-one assignments. It also uses no cosine threshold.

In [ ]:
centrality_bins = pd.qcut(
    source_details_cc['original_centrality_percentile'], 10, labels=False, duplicates='drop')
centrality_bins = pd.to_numeric(centrality_bins, errors='coerce') + 1
coverage_profile = (source_details_cc.assign(centrality_decile=centrality_bins)
                    .groupby(['condition', 'centrality_decile'], observed=True, as_index=False)
                    .agg(mean_coverage=('coverage_cosine', 'mean'),
                         selected_rate=('selected_by_any_summary_detail', 'mean'),
                         details=('original_detail_index', 'size')))
retention_details = source_details_cc.assign(
    retention_status=np.where(source_details_cc['selected_by_any_summary_detail'], 'NN-selected', 'not NN-selected'))
retention_story = (retention_details.groupby(
    ['story_key', 'record_id', 'condition', 'source_age', 'target_age', 'retention_status'], as_index=False)
    .agg(mean_original_centrality=('original_centrality', 'mean'), details=('original_detail_index', 'size')))
retention_wide = retention_story.pivot_table(
    index=['story_key', 'record_id', 'condition', 'source_age', 'target_age'],
    columns='retention_status', values='mean_original_centrality').reset_index()
if {'NN-selected', 'not NN-selected'} <= set(retention_wide.columns):
    retention_wide['retained_minus_not_retained_centrality'] = (
        retention_wide['NN-selected'] - retention_wide['not NN-selected'])
display(retention_story.groupby(['condition', 'retention_status'])
        .agg(stories=('story_key', 'nunique'), mean_centrality=('mean_original_centrality', 'mean')).reset_index())
if PLOTTING_AVAILABLE:
    fig, axes = plt.subplots(1, 3, figsize=(21, 5))
    sns.lineplot(data=coverage_profile, x='centrality_decile', y='mean_coverage', hue='condition', marker='o', ax=axes[0])
    sns.lineplot(data=coverage_profile, x='centrality_decile', y='selected_rate', hue='condition', marker='o', ax=axes[1])
    sns.regplot(data=retention_wide, x='source_age', y='retained_minus_not_retained_centrality',
                scatter_kws={'alpha': .25}, ax=axes[2])
    axes[0].set_title('Continuous semantic coverage')
    axes[1].set_title('Threshold-free NN-selection rate')
    axes[2].set_title('Centrality: selected minus not selected')
    for ax in axes[:2]:
        ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()

### Primary random control — does centrality predict coverage beyond chance?

Within each story-summary pair, randomly permute centrality values across the original details while leaving every coverage score unchanged. This preserves the story, summary, similarity matrix, and marginal distributions, but destroys the hypothesized correspondence between *which detail is central* and *which detail is covered*. The statistic is the mean within-story centrality–coverage correlation for each condition.

In [ ]:
def centrality_coverage_permutation(frame, permutations=1000, seed=20260902):
    rng = np.random.default_rng(seed)
    grouped = []
    for (_story_key, _record_id, condition), group in frame.groupby(['story_key', 'record_id', 'condition']):
        centrality = group['original_centrality'].to_numpy(float)
        coverage = group['coverage_cosine'].to_numpy(float)
        usable = np.isfinite(centrality) & np.isfinite(coverage)
        centrality, coverage = centrality[usable], coverage[usable]
        if len(centrality) < 3 or np.std(centrality) == 0 or np.std(coverage) == 0:
            continue
        observed = np.corrcoef(centrality, coverage)[0, 1]
        null = np.empty(permutations)
        for permutation_index in range(permutations):
            null[permutation_index] = np.corrcoef(rng.permutation(centrality), coverage)[0, 1]
        grouped.append((condition, observed, null))
    rows = []
    for condition in ['younger', 'actual', 'older']:
        selected = [(observed, null) for label, observed, null in grouped if label == condition]
        if not selected:
            continue
        observed = np.mean([item[0] for item in selected])
        null = np.mean(np.stack([item[1] for item in selected]), axis=0)
        rows.append({
            'condition': condition, 'n_story_summaries': len(selected),
            'observed_mean_centrality_coverage_correlation': observed,
            'null_low': np.quantile(null, .025), 'null_high': np.quantile(null, .975),
            'permutation_p': (1 + np.sum(np.abs(null) >= abs(observed))) / (permutations + 1),
        })
    return pd.DataFrame(rows)

continuous_randomization = centrality_coverage_permutation(source_details_cc)
continuous_randomization['p_fdr_bh'] = bh_adjust(continuous_randomization['permutation_p'])
display(continuous_randomization)
if PLOTTING_AVAILABLE and not continuous_randomization.empty:
    plot_frame = continuous_randomization.set_index('condition').reindex(['younger', 'actual', 'older']).dropna().reset_index()
    positions = np.arange(len(plot_frame))
    plt.figure(figsize=(8, 5))
    plt.vlines(positions, plot_frame['null_low'], plot_frame['null_high'],
               color='gray', linewidth=7, alpha=.5, label='permuted 95% null')
    plt.scatter(positions, plot_frame['observed_mean_centrality_coverage_correlation'],
                color='black', s=65, zorder=3, label='observed')
    plt.xticks(positions, plot_frame['condition'])
    plt.axhline(0, color='black', linestyle='--', linewidth=1)
    plt.ylabel('Mean within-story centrality–coverage correlation')
    plt.title('Does centrality predict semantic coverage beyond chance?')
    plt.legend()
    plt.tight_layout()

### Diagnostic random control for binary NN winners

For each story-summary pair, this diagnostic samples the same number of original details as the observed nearest-neighbor winners but chooses them randomly. It tests the behavior of the argmax matching rule, not continuous preservation. Because several summary details can select the same source detail and malformed fragments can become nearest neighbors, this result must not be used as the primary retention claim.

In [ ]:
def random_detail_selection_control(frame, permutations=1000, seed=20260902):
    rng = np.random.default_rng(seed)
    observed_by_condition = {condition: [] for condition in ['younger', 'actual', 'older']}
    null_by_condition = {condition: np.zeros(permutations) for condition in observed_by_condition}
    counts = {condition: 0 for condition in observed_by_condition}
    for (_story_key, _record_id, condition), group in frame.groupby(['story_key', 'record_id', 'condition']):
        if condition not in observed_by_condition:
            continue
        centrality = group['original_centrality'].to_numpy(float)
        selected = group['selected_by_any_summary_detail'].astype(bool).to_numpy()
        n_selected = int(selected.sum())
        if not (0 < n_selected < len(group)):
            continue
        observed_by_condition[condition].append(centrality[selected].mean() - centrality.mean())
        for permutation_index in range(permutations):
            sampled = rng.choice(len(group), size=n_selected, replace=False)
            null_by_condition[condition][permutation_index] += centrality[sampled].mean() - centrality.mean()
        counts[condition] += 1
    rows = []
    for condition in observed_by_condition:
        if not counts[condition]:
            continue
        observed = np.mean(observed_by_condition[condition])
        null = null_by_condition[condition] / counts[condition]
        rows.append({
            'condition': condition, 'n_story_summaries': counts[condition],
            'observed_centrality_enrichment': observed,
            'null_low': np.quantile(null, .025), 'null_high': np.quantile(null, .975),
            'permutation_p': (1 + np.sum(np.abs(null) >= abs(observed))) / (permutations + 1),
        })
    return pd.DataFrame(rows)

detail_randomization = random_detail_selection_control(source_details_cc)
detail_randomization['p_fdr_bh'] = bh_adjust(detail_randomization['permutation_p'])
display(detail_randomization)
if PLOTTING_AVAILABLE and not detail_randomization.empty:
    plot_frame = detail_randomization.set_index('condition').reindex(['younger', 'actual', 'older']).dropna().reset_index()
    plt.figure(figsize=(8, 5))
    positions = np.arange(len(plot_frame))
    plt.vlines(positions, plot_frame['null_low'], plot_frame['null_high'],
               color='gray', linewidth=6, alpha=.5, label='random-selection 95% null')
    plt.scatter(positions, plot_frame['observed_centrality_enrichment'],
                color='black', s=60, zorder=3, label='observed')
    plt.xticks(positions, plot_frame['condition'])
    plt.axhline(0, color='black', linestyle='--', linewidth=1)
    plt.ylabel('Selected-detail centrality minus story mean')
    plt.title('Observed centrality enrichment against random detail selection')
    plt.legend()
    plt.tight_layout()

### Worked nearest-neighbor mapping

Each point below is one summary detail. Its horizontal position is the original detail selected by nearest-neighbor cosine; its vertical position is the summary detail's narrative order. Color is match cosine. A monotonic diagonal indicates preserved chronology, while repeated horizontal positions indicate fusion or repeated grounding onto one source detail.

In [ ]:
example_counts = summary_details.groupby('record_id').size().sort_values()
example_record = example_counts.index[len(example_counts) // 2]
example_mapping = summary_details[summary_details['record_id'].eq(example_record)].sort_values('summary_detail_index')
display(example_mapping[[
    'summary_detail_index', 'summary_detail_text', 'grounded_original_detail_index',
    'grounded_original_detail_text', 'grounding_cosine', 'unique_original_detail_index', 'unique_match_cosine',
]])
if PLOTTING_AVAILABLE:
    plt.figure(figsize=(10, 6))
    sns.scatterplot(data=example_mapping, x='grounded_original_detail_index', y='summary_detail_index',
                    hue='grounding_cosine', size='grounding_cosine', palette='viridis', sizes=(50, 220))
    plt.title(f'Summary → original nearest-neighbor mapping: {example_record}')
    plt.xlabel('Selected original-detail index')
    plt.ylabel('Summary-detail index')
    plt.tight_layout()

## Primary representational Q — which details gain or lose weight after compression?

For every original detail, relative weight change is

`cos(original detail, whole summary) − cos(original detail, whole original)`.

Positive values indicate that the detail becomes more aligned with the compressed whole; negative values indicate representational downweighting. The main question is not whether the average changes, but whether original centrality predicts this change differently across target conditions and source ages.

In [ ]:
weight_profile = (source_details_cc.groupby(['story_key', 'record_id', 'condition', 'source_age', 'target_age'], as_index=False)
                  .agg(mean_weight_change=('original_detail_global_weight_change', 'mean'),
                       centrality_weight_correlation=('original_centrality', lambda s: np.nan)))
correlations = (source_details_cc.groupby(['story_key', 'record_id'], sort=False)
                .apply(lambda d: d['original_centrality'].corr(d['original_detail_global_weight_change']))
                .rename('centrality_weight_correlation').reset_index())
weight_profile = weight_profile.drop(columns='centrality_weight_correlation').merge(correlations, on=['story_key', 'record_id'])
if PLOTTING_AVAILABLE:
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    palette = {'younger': '#3b82f6', 'actual': '#4b5563', 'older': '#ef4444'}
    for condition in ['younger', 'actual', 'older']:
        frame = weight_profile[weight_profile['condition'].eq(condition)]
        sns.regplot(data=frame, x='source_age', y='mean_weight_change', scatter=False,
                    color=palette[condition], label=condition, ax=axes[0])
    sns.scatterplot(data=weight_profile, x='source_age', y='mean_weight_change',
                    hue='condition', palette=palette, alpha=.25, s=25, legend=False, ax=axes[0])
    sns.boxplot(data=weight_profile, x='condition', y='centrality_weight_correlation',
                order=['younger', 'actual', 'older'], palette=palette, ax=axes[1])
    axes[0].set_title('How mean detail reweighting changes with source age')
    axes[0].legend(title='Target condition')
    axes[1].set_title('Does centrality predict representational reweighting?')
    plt.tight_layout()

## Primary Q2 — do preservation and prioritization change across source age?

For each outcome, the target-condition coefficient tests a within-cohort shift at the centered source age; the condition × source-age term tests whether that shift changes developmentally. Models control token compression ratio, detail-count ratio, and original detail count so the younger condition's shorter outputs are not mistaken for a pure age effect. The primary outcomes are continuous coverage, centrality-selective coverage, and representational weight change. Binary NN selection is excluded from the primary family because the random-control analysis shows that argmax winners are not a valid retention definition.

In [ ]:
outcomes = [
    'mean_coverage_cosine', 'centrality_coverage_correlation',
    'mean_original_detail_global_weight_change',
]
if PLOTTING_AVAILABLE:
    fig, axes = plt.subplots(1, 3, figsize=(20, 5))
    for ax, outcome in zip(axes.flat, outcomes):
        for condition in ['younger', 'actual', 'older']:
            frame = story_cc[story_cc['condition'].eq(condition)]
            sns.regplot(data=frame, x='source_age', y=outcome, scatter=False,
                        color=palette[condition], label=condition, ax=ax)
        sns.scatterplot(data=story_cc, x='source_age', y=outcome, hue='condition', palette=palette,
                        alpha=.22, s=22, legend=False, ax=ax)
        ax.set_title(outcome.replace('_', ' ').title())
        ax.legend(title='Target condition')
    plt.tight_layout()

models = {}
developmental_tables = []
for outcome in outcomes:
    if STATSMODELS_AVAILABLE:
        frame = story_cc.dropna(subset=[outcome, 'source_age', 'condition']).copy()
        model = smf.ols(f'{outcome} ~ C(condition) * source_age + token_ratio + detail_ratio + n_original_details', data=frame)
        models[outcome] = model.fit(cov_type='cluster', cov_kwds={'groups': frame['story_key']})
        table = pd.DataFrame({
            'term': models[outcome].params.index, 'estimate': models[outcome].params.values,
            'clustered_se': models[outcome].bse.values, 't': models[outcome].tvalues.values,
            'p_value': models[outcome].pvalues.values, 'clusters': frame['story_key'].nunique(),
            'observations': len(frame),
        })
    else:
        models[outcome] = clustered_age_condition_regression(
            story_cc, outcome, controls=('token_ratio', 'detail_ratio', 'n_original_details'))
        table = models[outcome].copy()
    table.insert(0, 'outcome', outcome)
    developmental_tables.append(table)
developmental_tests = pd.concat(developmental_tables, ignore_index=True)
test_mask = developmental_tests['term'].str.contains('condition|source_age', case=False, regex=True)
developmental_tests.loc[test_mask, 'p_fdr_bh'] = bh_adjust(
    developmental_tests.loc[test_mask, 'p_value'])
display(developmental_tests.loc[test_mask].sort_values(['outcome', 'term']))

### Paired within-story condition tests

These tests compare all three target conditions within the same complete-case stories. The story—not the detail—is the sampling unit. Wilcoxon signed-rank tests avoid assuming normally distributed differences, and Benjamini–Hochberg correction controls false discovery rate across the prespecified outcomes and contrasts.

In [ ]:
from scipy.stats import wilcoxon

paired_rows = []
story_condition = story_cc.groupby(['story_key', 'condition'], as_index=False)[outcomes].mean()
for outcome in outcomes:
    wide = story_condition.pivot(index='story_key', columns='condition', values=outcome)
    for comparison, reference in [('older', 'actual'), ('younger', 'actual'), ('older', 'younger')]:
        if not {reference, comparison} <= set(wide.columns):
            continue
        pair = wide[[reference, comparison]].dropna()
        differences = pair[comparison] - pair[reference]
        statistic, p_value = wilcoxon(differences) if len(pair) and np.any(differences != 0) else (np.nan, np.nan)
        paired_rows.append({
            'outcome': outcome, 'contrast': f'{comparison} - {reference}', 'n_stories': len(pair),
            'mean_difference': differences.mean(), 'median_difference': differences.median(),
            'wilcoxon_statistic': statistic, 'p_value': p_value,
        })
paired_tests = pd.DataFrame(paired_rows)
usable = paired_tests['p_value'].notna()
p_values = paired_tests.loc[usable, 'p_value'].to_numpy(float)
order = np.argsort(p_values)
ranked = p_values[order] * len(p_values) / np.arange(1, len(p_values) + 1)
ranked = np.minimum.accumulate(ranked[::-1])[::-1]
adjusted = np.empty_like(ranked)
adjusted[order] = np.clip(ranked, 0, 1)
paired_tests.loc[usable, 'p_fdr_bh'] = adjusted
display(paired_tests.sort_values(['outcome', 'contrast']))

### Does the same three-year extrapolation beyond the source-age bounds change more rapidly for younger stories?

For each complete story and outcome, define the local target-age response as `g = (older - younger) / (older target age - younger target age)`. This is a centered finite-difference estimate of how the compression outcome changes per target-age year. The denominator is read from the data because younger is three years below the source range's lower bound and older is three years above its upper bound; for a multi-year source range, their distance from the midpoint is therefore greater than three. Curvature is `(older - 2*actual + younger) / half_span²`; it tests whether the actual-age summary lies on the straight line between the two steered summaries.

The developmental null is tested by permuting the **source-age label across complete story triplets**, never by separating a story's younger/actual/older summaries. The primary continuous statistic is the Spearman association between source age and `g`. The prespecified window statistic compares mean `g` for source ages 3–6 versus 8–11 and is reported only when both windows have adequate support. A shared permutation is used across outcomes, and max-|T| provides familywise correction.

`token_ratio` and `detail_ratio` are length/count controls. They are not semantic similarity: token compression is summary tokens divided by original tokens, whereas semantic preservation is measured by cosine-based detail coverage.

In [ ]:
from scipy.stats import rankdata

def correlation_t(x, y):
    usable = np.isfinite(x) & np.isfinite(y)
    x, y = np.asarray(x)[usable], np.asarray(y)[usable]
    if len(x) < 4 or np.std(x) == 0 or np.std(y) == 0:
        return np.nan
    correlation = np.corrcoef(rankdata(x), rankdata(y))[0, 1]
    return correlation * np.sqrt((len(x) - 2) / max(1 - correlation ** 2, 1e-12))

def welch_t(young, old):
    young, old = np.asarray(young, float), np.asarray(old, float)
    young, old = young[np.isfinite(young)], old[np.isfinite(old)]
    if len(young) < 2 or len(old) < 2:
        return np.nan
    denominator = np.sqrt(np.var(young, ddof=1) / len(young) + np.var(old, ddof=1) / len(old))
    return (np.mean(young) - np.mean(old)) / denominator if denominator > 0 else np.nan

story_age = (story_cc[['story_key', 'source_age']].dropna()
             .drop_duplicates('story_key').set_index('story_key')['source_age'])
target_age_wide = (story_cc.groupby(['story_key', 'condition'], as_index=False)['target_age'].mean()
                   .pivot(index='story_key', columns='condition', values='target_age'))
local_rows = []
for outcome in outcomes:
    wide = story_condition.pivot(index='story_key', columns='condition', values=outcome)
    wide = wide.reindex(columns=['younger', 'actual', 'older']).dropna()
    for story_key, values in wide.iterrows():
        target_ages = target_age_wide.loc[story_key, ['younger', 'actual', 'older']]
        target_span = target_ages['older'] - target_ages['younger']
        half_span = target_span / 2.0
        if not np.isfinite(target_span) or target_span <= 0:
            continue
        local_rows.append({
            'story_key': story_key, 'outcome': outcome, 'source_age': story_age.get(story_key, np.nan),
            'younger_target_age': target_ages['younger'], 'actual_target_age': target_ages['actual'],
            'older_target_age': target_ages['older'], 'half_span_years': half_span,
            'local_response_per_year': (values['older'] - values['younger']) / target_span,
            'local_curvature': (values['older'] - 2 * values['actual'] + values['younger']) / half_span ** 2,
        })
local_age_response = pd.DataFrame(local_rows).dropna(subset=['source_age'])
support = (local_age_response.drop_duplicates(['story_key', 'source_age'])
           .assign(age_window=lambda d: np.select(
               [d['source_age'].between(3, 6), d['source_age'].between(8, 11)],
               ['3–6', '8–11'], default='outside'))
           .groupby('age_window', as_index=False).agg(stories=('story_key', 'nunique')))
display(support)
display(local_age_response.groupby('outcome').agg(
    stories=('story_key', 'nunique'), source_age_min=('source_age', 'min'),
    source_age_max=('source_age', 'max'), mean_local_response=('local_response_per_year', 'mean'),
    mean_curvature=('local_curvature', 'mean')).reset_index())

In [ ]:
def source_age_block_permutation(frame, permutations=10000, seed=20260902, minimum_window_stories=10):
    rng = np.random.default_rng(seed)
    story_ages = (frame.groupby('story_key', as_index=False)['source_age'].mean()
                  .sort_values('story_key'))
    story_keys = story_ages['story_key'].to_numpy()
    ages = story_ages['source_age'].to_numpy(float)
    age_position = {story_key: index for index, story_key in enumerate(story_keys)}
    tests = []
    for outcome, outcome_frame in frame.groupby('outcome'):
        row_positions = np.array([age_position[key] for key in outcome_frame['story_key']])
        values = outcome_frame['local_response_per_year'].to_numpy(float)
        observed_ages = ages[row_positions]
        tests.append({'outcome': outcome, 'test': 'source-age Spearman trend',
                      'observed_t': correlation_t(observed_ages, values),
                      'effect': np.corrcoef(rankdata(observed_ages), rankdata(values))[0, 1],
                      'n_stories': len(values), 'row_positions': row_positions, 'values': values})
        young = (observed_ages >= 3) & (observed_ages <= 6)
        old = (observed_ages >= 8) & (observed_ages <= 11)
        if young.sum() >= minimum_window_stories and old.sum() >= minimum_window_stories:
            tests.append({'outcome': outcome, 'test': 'mean local response: age 3–6 minus 8–11',
                          'observed_t': welch_t(values[young], values[old]),
                          'effect': np.mean(values[young]) - np.mean(values[old]),
                          'n_stories': len(values), 'n_young': int(young.sum()), 'n_old': int(old.sum()),
                          'row_positions': row_positions, 'values': values})
    null_statistics = np.full((permutations, len(tests)), np.nan)
    for permutation_index in range(permutations):
        permuted_ages = rng.permutation(ages)
        for test_index, test in enumerate(tests):
            test_ages = permuted_ages[test['row_positions']]
            if test['test'] == 'source-age Spearman trend':
                null_statistics[permutation_index, test_index] = correlation_t(test_ages, test['values'])
            else:
                young = (test_ages >= 3) & (test_ages <= 6)
                old = (test_ages >= 8) & (test_ages <= 11)
                null_statistics[permutation_index, test_index] = welch_t(
                    test['values'][young], test['values'][old])
    finite_for_max = np.where(np.isfinite(null_statistics), np.abs(null_statistics), -np.inf)
    max_abs_null = np.max(finite_for_max, axis=1)
    rows = []
    for test_index, test in enumerate(tests):
        null = null_statistics[:, test_index]
        null = null[np.isfinite(null)]
        observed = test['observed_t']
        row = {key: value for key, value in test.items() if key not in {'row_positions', 'values'}}
        row['permutation_p'] = (1 + np.sum(np.abs(null) >= abs(observed))) / (len(null) + 1)
        row['p_fwer_maxT'] = (1 + np.sum(max_abs_null >= abs(observed))) / (len(max_abs_null) + 1)
        rows.append(row)
    return pd.DataFrame(rows)

local_age_permutation = source_age_block_permutation(local_age_response)
display(local_age_permutation.sort_values(['test', 'outcome']))
if PLOTTING_AVAILABLE:
    fig, axes = plt.subplots(1, len(outcomes), figsize=(20, 5), constrained_layout=True)
    for ax, outcome in zip(np.atleast_1d(axes), outcomes):
        frame = local_age_response[local_age_response['outcome'].eq(outcome)]
        sns.regplot(data=frame, x='source_age', y='local_response_per_year', order=2,
                    scatter_kws={'alpha': .3, 's': 24}, line_kws={'color': 'black'}, ax=ax)
        ax.axhline(0, color='gray', linestyle='--', linewidth=1)
        ax.axvspan(3, 6, color='#3b82f6', alpha=.08)
        ax.axvspan(8, 11, color='#ef4444', alpha=.08)
        ax.set_title(outcome.replace('_', ' ').title())
        ax.set_ylabel('(older − younger) / target-age span')
    plt.show()

### Does representational age-response align with behavioral detail preservation?

This test links levels rather than merely asking whether each level separately changes with age. For every story, the representation response is the multivariate PCA displacement `(older - younger) / target-age span`; the behavioral response is the corresponding change in detail coverage, centrality-selectivity, or detail weight. Both are residualized for source age, the age-response of token/detail compression ratios, and original detail count. Linear centered-kernel alignment then tests whether stories with similar residual representational responses also have similar residual behavioral responses. Story labels are permuted as blocks and max-statistic correction covers all layers and behavioral outcomes.

In [ ]:
def residualize_matrix(values, design):
    values = np.asarray(values, float)
    design = np.asarray(design, float)
    design = np.column_stack([np.ones(len(design)), design])
    return values - design @ (np.linalg.pinv(design) @ values)

def centered_linear_cka(left, right):
    left = np.asarray(left, float)
    right = np.asarray(right, float).reshape(len(left), -1)
    left = left - left.mean(axis=0, keepdims=True)
    right = right - right.mean(axis=0, keepdims=True)
    cross = left.T @ right
    denominator = np.linalg.norm(left.T @ left) * np.linalg.norm(right.T @ right)
    return float(np.sum(cross ** 2) / denominator) if denominator > 0 else np.nan

ratio_wide = story_cc.pivot_table(index='story_key', columns='condition',
                                  values=['token_ratio', 'detail_ratio'], aggfunc='mean')
control_rows = []
for story_key in valid_story_keys:
    span = target_lookup[(story_key, 'older')] - target_lookup[(story_key, 'younger')]
    story_frame = story_cc[story_cc['story_key'].eq(story_key)]
    control_rows.append({
        'story_key': story_key, 'source_age': source_lookup[story_key],
        'token_ratio_response': (ratio_wide.loc[story_key, ('token_ratio', 'older')] -
                                 ratio_wide.loc[story_key, ('token_ratio', 'younger')]) / span,
        'detail_ratio_response': (ratio_wide.loc[story_key, ('detail_ratio', 'older')] -
                                  ratio_wide.loc[story_key, ('detail_ratio', 'younger')]) / span,
        'n_original_details': story_frame['n_original_details'].mean(),
    })
control_frame = pd.DataFrame(control_rows).set_index('story_key')
behavior_wide = local_age_response.pivot(index='story_key', columns='outcome',
                                         values='local_response_per_year')
common_keys = [key for key in valid_story_keys if key in behavior_wide.index and key in control_frame.index]
common_keys = [key for key in common_keys if behavior_wide.loc[key, outcomes].notna().all() and
               control_frame.loc[key].notna().all()]
controls = control_frame.loc[common_keys, ['source_age', 'token_ratio_response',
                                           'detail_ratio_response', 'n_original_details']].to_numpy(float)
controls = (controls - controls.mean(axis=0)) / np.where(controls.std(axis=0) > 0, controls.std(axis=0), 1)
story_position = {key: index for index, key in enumerate(valid_story_keys)}
selected_positions = np.array([story_position[key] for key in common_keys])
behavior_residuals = {outcome: residualize_matrix(behavior_wide.loc[common_keys, outcome].to_numpy(float), controls)
                      for outcome in outcomes}
representation_residuals = {int(layer): residualize_matrix(local_vectors[int(layer)][selected_positions], controls)
                            for layer in raw_layers}
alignment_rows = []
for layer in raw_layers:
    for outcome in outcomes:
        alignment_rows.append({'layer': int(layer), 'outcome': outcome,
                               'residual_linear_cka': centered_linear_cka(
                                   representation_residuals[int(layer)], behavior_residuals[outcome]),
                               'n_stories': len(common_keys)})
behavior_representation_alignment = pd.DataFrame(alignment_rows)
rng = np.random.default_rng(20260902)
alignment_null = np.empty((PCA_PERMUTATIONS, len(alignment_rows)))
for permutation_index in range(PCA_PERMUTATIONS):
    order = rng.permutation(len(common_keys))
    test_index = 0
    for layer in raw_layers:
        for outcome in outcomes:
            alignment_null[permutation_index, test_index] = centered_linear_cka(
                representation_residuals[int(layer)], behavior_residuals[outcome][order])
            test_index += 1
alignment_null_max = np.nanmax(alignment_null, axis=1)
behavior_representation_alignment['permutation_p_uncorrected'] = [
    (1 + np.sum(alignment_null[:, index] >= value)) / (PCA_PERMUTATIONS + 1)
    for index, value in enumerate(behavior_representation_alignment['residual_linear_cka'])]
behavior_representation_alignment['p_fwer_maxT'] = [
    (1 + np.sum(alignment_null_max >= value)) / (PCA_PERMUTATIONS + 1)
    for value in behavior_representation_alignment['residual_linear_cka']]
alignment_omnibus_p = (1 + np.sum(alignment_null_max >= behavior_representation_alignment['residual_linear_cka'].max())) / (PCA_PERMUTATIONS + 1)
print('Behavior–representation alignment omnibus p:', alignment_omnibus_p)
display(behavior_representation_alignment.sort_values(['outcome', 'layer']))
if PLOTTING_AVAILABLE:
    sns.lineplot(data=behavior_representation_alignment, x='layer', y='residual_linear_cka',
                 hue='outcome', marker='o')
    plt.title(f'Residual behavior–representation alignment (omnibus p={alignment_omnibus_p:.3g})')
    plt.ylabel('Linear CKA')
    plt.show()

## Q5 — Are results robust to matching and representation choices?

Three prespecified checks prevent the result from depending on one arbitrary choice:

- Compare nearest-neighbor `grounding_cosine` with `unique_match_cosine`.
- Repeat the HPC reducer using `--representation mean` as a pooling control.
- Repeat the complete notebook for Qwen3-8B and compare effect direction on the shared story set.

In [ ]:
robustness = (summary_details.groupby(['story_key', 'record_id', 'condition'], as_index=False)
              .agg(nearest_neighbor=('grounding_cosine', 'mean'), unique_matching=('unique_match_cosine', 'mean')))
display(robustness.groupby('condition')[['nearest_neighbor', 'unique_matching']].agg(['count', 'mean', 'std']))
sns.scatterplot(data=robustness, x='nearest_neighbor', y='unique_matching', hue='condition', alpha=.4)
limits = [robustness[['nearest_neighbor', 'unique_matching']].min().min(),
          robustness[['nearest_neighbor', 'unique_matching']].max().max()]
plt.plot(limits, limits, '--', color='black')
plt.title('Nearest-neighbor versus unique matching')
plt.show()

### Cross-model representational agreement

The same original and GPT-generated summary texts are encoded by GPT-OSS and Qwen. Agreement means the two models assign similar compression geometry to the same story-summary pair; disagreement is evidence that stable model behavior/text does not uniquely determine internal representation. This is the direct analysis motivated by *Not all solutions are created equal*, without claiming that these two pretrained models occupy the paper's analytically defined solution manifold.

In [ ]:
model_paths = {
    'gpt_oss_20b': ROOT / 'mechanistic_interpretability/outputs/02_representation_extraction/analysis/gpt_oss_20b/story_representation_compression_metrics.csv',
    'qwen3_8b': ROOT / 'mechanistic_interpretability/outputs/02_representation_extraction/analysis/qwen3_8b/story_representation_compression_metrics.csv',
}
if all(path.exists() for path in model_paths.values()):
    gpt_metrics = pd.read_csv(model_paths['gpt_oss_20b'], low_memory=False)
    qwen_metrics = pd.read_csv(model_paths['qwen3_8b'], low_memory=False)
    comparison_outcomes = [
        'mean_coverage_cosine', 'proportion_original_details_selected',
        'centrality_coverage_correlation', 'mean_original_detail_global_weight_change',
    ]
    shared = gpt_metrics.merge(qwen_metrics, on=['record_id', 'story_key', 'condition'], suffixes=('_gpt', '_qwen'))
    agreement_rows = []
    for outcome in comparison_outcomes:
        agreement_rows.append({
            'outcome': outcome, 'shared_summaries': len(shared),
            'pearson': shared[f'{outcome}_gpt'].corr(shared[f'{outcome}_qwen'], method='pearson'),
            'spearman': shared[f'{outcome}_gpt'].corr(shared[f'{outcome}_qwen'], method='spearman'),
        })
    display(pd.DataFrame(agreement_rows))
    if PLOTTING_AVAILABLE:
        fig, axes = plt.subplots(2, 2, figsize=(14, 12))
        for ax, outcome in zip(axes.flat, comparison_outcomes):
            sns.scatterplot(data=shared, x=f'{outcome}_gpt', y=f'{outcome}_qwen',
                            hue='condition', alpha=.3, s=20, ax=ax)
            ax.set_title(outcome.replace('_', ' ').title())
        plt.tight_layout()
else:
    print('Cross-model comparison will appear after both reducer jobs finish.')

## Q6 — Does narrative context change how a detail is represented?

For GPT-OSS, the isolated-detail control encodes the same wording without its story. `contextualization_shift = 1 − cosine(contextual, isolated)` measures how strongly narrative context changes that detail. This is not deletion influence: it directly compares the same detail with and without surrounding context.

In [ ]:
context_data = source_details.dropna(subset=['contextualization_shift']).copy()
if context_data.empty:
    print('No isolated-detail controls were supplied to the HPC reducer.')
else:
    context_story = (context_data.groupby(['story_key', 'record_id', 'condition', 'source_age'], as_index=False)
                     .agg(mean_contextualization_shift=('contextualization_shift', 'mean'),
                          centrality_context_correlation=('original_centrality',
                              lambda s: np.nan)))
    correlations = (context_data.groupby(['story_key', 'record_id'], sort=False)
                    .apply(lambda d: d['original_centrality'].corr(d['contextualization_shift']))
                    .rename('centrality_context_correlation').reset_index())
    context_story = context_story.drop(columns='centrality_context_correlation').merge(correlations, on=['story_key', 'record_id'])
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sns.regplot(data=context_story, x='source_age', y='mean_contextualization_shift', ax=axes[0])
    sns.boxplot(data=context_story, x='condition', y='centrality_context_correlation', ax=axes[1])
    axes[0].set_title('Narrative contextualization by source age')
    axes[1].set_title('Does context reshape central details differently?')
    plt.tight_layout()

## Interpretation checklist

Read results in this order:

1. **Amount:** do paired conditions differ in `mean_coverage_cosine`?
2. **Prioritization:** do they differ in `centrality_coverage_correlation`, and does the centrality-permutation null reject chance alignment?
3. **Development:** do the condition × source-age terms survive familywise FDR correction?
4. **Representation:** do central details gain or lose whole-summary weight in the same direction?
5. **Secondary layer evidence:** at which layers do paired whole-text contrasts emerge, and do their source-age slopes survive correction?
6. **Robustness:** does the direction replicate with isolated semantic matching and Qwen?

Do not treat binary NN winners as retained details, do not interpret unadjusted coefficient p-values selectively, and do not generalize beyond the observed complete-case source-age range.